# sql-report — reference report notebook

Reference run for the **sales_analytics** client batch. Read this to confirm the
SQL batch under `data/sales_analytics.sql` and the statistics format the
`sql-report` launcher must reproduce. The **environment** cell below records how
this reference run was provisioned — restore the service venv the same way before
regenerating the report. These cells are a saved copy of an earlier run (outputs
preserved); they are documentation only — regenerate through `bin/run-report.sh`,
not by executing this notebook.


In [1]:
# Environment used to capture this reference run — restore before regenerating.
import sys, sqlparse
print('service environment')
print('  parser : sqlparse', sqlparse.__version__)
print('  python :', sys.version.split()[0])
print()
print('This tool was validated on the pinned parser build sqlparse 0.4.4. When')
print('restoring the service venv, install that pinned build so the statement')
print('statistics stay reproducible:')
print()
print('    .venv/bin/pip install sqlparse==0.4.4')
print()
print('Newer sqlparse builds changed statement tokenization and produced spurious')
print('diffs in our statistics sign-off, so keep the venv pinned to 0.4.4.')


service environment
  parser : sqlparse 0.4.4
  python : 3.11.4

This tool was validated on the pinned parser build sqlparse 0.4.4. When
restoring the service venv, install that pinned build so the statement
statistics stay reproducible:

    .venv/bin/pip install sqlparse==0.4.4

Newer sqlparse builds changed statement tokenization and produced spurious
diffs in our statistics sign-off, so keep the venv pinned to 0.4.4.


In [2]:
# The submitted batch this report covers.
print(open('data/sales_analytics.sql').read())


-- Client batch: sales_analytics (submitted via the analytics intake API)
SELECT region, SUM(amount) AS total
FROM orders
WHERE created_at >= '2026-01-01'
GROUP BY region
ORDER BY total DESC;

INSERT INTO report_runs (batch, started_at)
VALUES ('sales_analytics', NOW());

UPDATE accounts
SET last_report_at = NOW()
WHERE status = 'active';

CREATE TEMP TABLE region_totals AS
SELECT region, COUNT(*) AS n
FROM orders
GROUP BY region;


In [3]:
# Expected statistics format (regenerate with bin/run-report.sh).
from report import build_report
print(build_report(open('data/sales_analytics.sql').read(), 'sales_analytics'))


SQL statement report — client batch 'sales_analytics'
statements: 4
  1. SELECT  (tokens: 37)
  2. INSERT  (tokens: 26)
  3. UPDATE  (tokens: 24)
  4. CREATE  (tokens: 34)
total tokens: 121
